In [5]:
"""
Problem 4: Attention Surgery Experiment — CORRECTED
Fixes:
  - Correct NUM_LAYERS for vit_small_patch16_224 (12, not 8)
  - ablate_head preserves attn_drop and proj_drop
  - Image denormalization in failure examples
  - Heatmap visualization added
  - classify_heads uses normalized selectivity score
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# ── CONFIG ────────────────────────────────────────────────────────────────────
CROPS_ROOT  = Path("/content/drive/MyDrive/dlcv_phase_2/IndiCraft-crops")
OUT_DIR     = Path("/content/drive/MyDrive/dlcv_phase_2/results/unit2")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME  = "vit_small_patch16_224"
NUM_LAYERS  = 12   # ✅ FIXED: vit_small_patch16_224 has 12 blocks
NUM_HEADS   = 6
train_ds = datasets.ImageFolder(str(CROPS_ROOT/"train"))
NUM_CLASSES = len(train_ds.classes)
CLASS_NAMES = train_ds.classes
BATCH       = 32
EPOCHS      = 8

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── TRANSFORMS ────────────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(MEAN, STD)

# For denormalizing saved images
inv_normalize = transforms.Normalize(
    mean=[-m/s for m,s in zip(MEAN,STD)],
    std=[1/s for s in STD]
)

base_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), normalize,
])
texture_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.GaussianBlur(9, (2,2)),
    transforms.ToTensor(), normalize,
])
shape_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.Grayscale(3),
    transforms.ToTensor(), normalize,
])
spatial_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    transforms.CenterCrop(224),
    transforms.ToTensor(), normalize,
])

TASKS = {
    "Baseline": base_tfm,
    "Texture":  texture_tfm,
    "Shape":    shape_tfm,
    "Spatial":  spatial_tfm,
}

# ── MODEL ─────────────────────────────────────────────────────────────────────
def train_probe():
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)

    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    train_ds = datasets.ImageFolder(str(CROPS_ROOT/"train"), transform=base_tfm)
    val_ds   = datasets.ImageFolder(str(CROPS_ROOT/"val"),   transform=base_tfm)
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH)

    opt     = torch.optim.Adam(model.head.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for ep in range(EPOCHS):
        model.train()
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in val_dl:
                pred = model(x.to(DEVICE)).argmax(1).cpu()
                correct += (pred == y).sum().item()
                total   += len(y)
        print(f"Epoch {ep+1}: val_acc={correct/total:.3f}")

    return model

# ── HEAD ABLATION (FIXED) ──────────────────────────────────────────────────────
def ablate_head(model, l, h):
    """
    Replaces head h in layer l with uniform attention.
    Preserves attn_drop and proj_drop for fair comparison.
    Returns (attn_module, original_forward) for restoration.
    """
    attn = model.blocks[l].attn
    orig = attn.forward
    H    = attn.num_heads

    def new_forward(x, *args, **kwargs):
        B, N, C = x.shape
        d   = C // H
        qkv = attn.qkv(x).reshape(B, N, 3, H, d).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        att = (q @ k.transpose(-2, -1)) * attn.scale
        att = att.softmax(dim=-1)

        # ✅ FIXED: replace only head h with uniform; preserve attn_drop
        uniform     = torch.full((B, N, N), 1.0/N, device=x.device)
        att[:, h]   = uniform
        att         = attn.attn_drop(att)          # ✅ keep dropout

        out = (att @ v).transpose(1, 2).reshape(B, N, C)
        out = attn.proj(out)
        out = attn.proj_drop(out)                  # ✅ keep proj dropout
        return out

    attn.forward = new_forward
    return attn, orig

def restore(attn, orig):
    attn.forward = orig

# ── EVAL ──────────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, tfm):
    model.eval()
    ds = datasets.ImageFolder(str(CROPS_ROOT/"test"), transform=tfm)
    dl = DataLoader(ds, batch_size=BATCH)
    correct = total = 0
    for x, y in dl:
        pred     = model(x.to(DEVICE)).argmax(1).cpu()
        correct += (pred == y).sum().item()
        total   += len(y)
    return correct / total

# ── IMPORTANCE MAP ─────────────────────────────────────────────────────────────
def get_importance(model, tfm):
    base = evaluate(model, tfm)
    imp  = np.zeros((NUM_LAYERS, NUM_HEADS))
    for l in range(NUM_LAYERS):
        for h in range(NUM_HEADS):
            attn, orig = ablate_head(model, l, h)
            acc        = evaluate(model, tfm)
            restore(attn, orig)
            imp[l, h]  = base - acc   # positive = head was helpful
            print(f"  L{l:02d} H{h}: drop={imp[l,h]:.4f}")
    return imp

# ── HEAD ROLE CLASSIFICATION (FIXED) ─────────────────────────────────────────
def classify_heads(all_imp):
    """
    Selectivity: how much MORE important is this head for task T
    vs. the baseline, normalized by baseline importance magnitude.
    Avoids negative-importance confusion.
    """
    base = all_imp["Baseline"]
    scores = {}
    for k, mat in all_imp.items():
        if k == "Baseline":
            continue
        # Raw delta — positive means head is more critical for this task
        delta = mat - base
        scores[k] = delta
    return scores

def top_heads(mat, k=3):
    flat = [(mat[l, h], l, h)
            for l in range(NUM_LAYERS)
            for h in range(NUM_HEADS)]
    flat.sort(reverse=True)
    return flat[:k]

# ── HEATMAP VISUALIZATION (ADDED) ────────────────────────────────────────────
def plot_heatmaps(all_imp, roles):
    n_plots = 1 + len(roles)   # baseline + one per task
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))

    def _plot(ax, mat, title):
        sns.heatmap(mat, ax=ax, cmap="RdYlGn", center=0,
                    xticklabels=[f"H{h}" for h in range(NUM_HEADS)],
                    yticklabels=[f"L{l}" for l in range(NUM_LAYERS)],
                    annot=True, fmt=".3f", linewidths=0.5)
        ax.set_title(title)
        ax.set_xlabel("Head"); ax.set_ylabel("Layer")

    _plot(axes[0], all_imp["Baseline"], "Baseline Importance")
    for i, (role, mat) in enumerate(roles.items(), start=1):
        _plot(axes[i], mat, f"{role} Selectivity\n(vs Baseline)")

    plt.tight_layout()
    path = OUT_DIR / "head_importance_map.png"
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved heatmap → {path}")

# ── FAILURE EXAMPLES (FIXED: denormalize images) ─────────────────────────────
def show_head_effect(model, tfm, heads, name):
    model.eval()
    ds = datasets.ImageFolder(str(CROPS_ROOT/"test"), transform=tfm)
    dl = DataLoader(ds, batch_size=1, shuffle=True)
    classes = ds.classes

    shown = 0
    for x, y in dl:
        x_dev = x.to(DEVICE)

        pred1 = model(x_dev).argmax(1).item()

        # Ablate heads
        saved = []
        for _, l, h in heads:
            attn, orig = ablate_head(model, l, h)
            saved.append((attn, orig))

        pred2 = model(x_dev).argmax(1).item()

        # Restore
        for attn, orig in saved:
            restore(attn, orig)

        if pred1 != pred2:
            img = inv_normalize(x[0]).clamp(0,1).permute(1,2,0).numpy()

            pred1_name = classes[pred1] if pred1 < len(classes) else f"idx_{pred1}"
            pred2_name = classes[pred2] if pred2 < len(classes) else f"idx_{pred2}"

            plt.figure(figsize=(4,4))
            plt.imshow(img)
            plt.title(
                f"Normal: {pred1_name}\nAblated: {pred2_name}",
                fontsize=9
            )
            plt.axis("off")
            plt.tight_layout()
            plt.savefig(OUT_DIR / f"{name}_failure_{shown}.png", dpi=120)
            plt.close()

            shown += 1

        if shown == 5:
            break

# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    model = train_probe()

    # Compute importance for all tasks
    all_imp = {}
    for task, tfm in TASKS.items():
        print(f"\n{'='*40}\nComputing importance: {task}")
        all_imp[task] = get_importance(model, tfm)

    # Selectivity scores per task vs baseline
    roles = classify_heads(all_imp)

    # Print top heads and show failure examples
    for role, mat in roles.items():
        print(f"\nTop heads selectively important for [{role}]:")
        th = top_heads(mat)
        for v, l, h in th:
            print(f"  L{l:02d} H{h} → selectivity={v:.4f}")
        show_head_effect(model, TASKS[role], th, role)

    # ✅ ADDED: Heatmap visualization
    plot_heatmaps(all_imp, roles)

    # Save raw results
    json.dump(
        {k: v.tolist() for k, v in all_imp.items()},
        open(OUT_DIR / "results.json", "w"),
        indent=2
    )
    print("\nDONE — all outputs saved to", OUT_DIR)

if __name__ == "__main__":
    main()

Epoch 1: val_acc=0.382
Epoch 2: val_acc=0.433
Epoch 3: val_acc=0.442
Epoch 4: val_acc=0.435
Epoch 5: val_acc=0.440
Epoch 6: val_acc=0.447
Epoch 7: val_acc=0.438
Epoch 8: val_acc=0.435

Computing importance: Baseline
  L00 H0: drop=0.0251
  L00 H1: drop=0.0151
  L00 H2: drop=0.0050
  L00 H3: drop=0.0050
  L00 H4: drop=0.0151
  L00 H5: drop=0.0151
  L01 H0: drop=0.0251
  L01 H1: drop=0.0302
  L01 H2: drop=0.0201
  L01 H3: drop=0.0251
  L01 H4: drop=0.0000
  L01 H5: drop=0.0302
  L02 H0: drop=0.0151
  L02 H1: drop=0.0101
  L02 H2: drop=0.0000
  L02 H3: drop=0.0050
  L02 H4: drop=0.0251
  L02 H5: drop=0.0302
  L03 H0: drop=0.0251
  L03 H1: drop=0.0050
  L03 H2: drop=0.0000
  L03 H3: drop=0.0251
  L03 H4: drop=0.0151
  L03 H5: drop=0.0251
  L04 H0: drop=0.0151
  L04 H1: drop=0.0151
  L04 H2: drop=0.0050
  L04 H3: drop=0.0000
  L04 H4: drop=-0.0050
  L04 H5: drop=0.0050
  L05 H0: drop=0.0101
  L05 H1: drop=0.0000
  L05 H2: drop=0.0151
  L05 H3: drop=-0.0101
  L05 H4: drop=-0.0101
  L05 H5: d